In [9]:
import os
import re
import pandas as pd

def parse_log(file_path):
    """Извлекает ключевые пары параметр-значение из .log файла"""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()

    patterns = {
        'model': r'Using model[:=]\s*([A-Za-z0-9_\-]+)',
        'backbone': r'Using backbone[:=]\s*([A-Za-z0-9_\-]+)',
        'tile size': r'Using tile size[:=]\s*(\d+)',
        'stride': r'Using stride[:=]\s*(\d+)',
        'batch size': r'Using batch size[:=]\s*(\d+)',
        'num workers': r'Using num workers[:=]\s*(\d+)',
        'num epochs': r'Using num epochs[:=]\s*(\d+)',
        'learning rate': r'Using learning rate[:=]\s*([0-9.eE-]+)',
        'positive weight': r'Using positive weight[:=]\s*([0-9.eE-]+)',
        'reduction': r'Using reduction[:=]\s*([A-Za-z0-9_\-]+)',
        'optimizer': "Adam",
        'accuracy': r'[Aa]ccuracy[:=]\s*([0-9.]+)',
        'precision': r'[Pp]recision[:=]\s*([0-9.]+)',
        'recall': r'[Rr]ecall[:=]\s*([0-9.]+)',
        'f1': r'[Ff]1[:=]\s*([0-9.]+)',
        'iou': r'[Ii]ou[:=]\s*([0-9.]+)',
        'specificity': r'[Ss]pecificity[:=]\s*([0-9.]+)',
    }

    result = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            result[key] = match.group(1)

    return result

def scan_experiments(base_dir='experiments'):
    """Сканирует все подпапки и собирает данные из training.log и evaluation.log"""
    records = []
    for root, dirs, files in os.walk(base_dir):
        if 'training.log' in files or 'evaluation.log' in files:
            model_name = os.path.basename(root)
            entry = {'model_name': model_name}
            for log in ['training.log', 'evaluation.log']:
                path = os.path.join(root, log)
                if os.path.exists(path):
                    parsed = parse_log(path)
                    for k, v in parsed.items():
                        entry[f"{log.split('.')[0]}_{k}"] = v
            records.append(entry)
    return pd.DataFrame(records)

df = scan_experiments('../experiments')

df_sorted = df.sort_values(by='evaluation_iou', ascending=False)
display(df_sorted)
print(df_sorted['model_name'].iloc[0])

df_sorted.to_csv('experiments.csv', index=False)

,model_name,training_model,training_backbone,training_tile size,training_stride,training_batch size,training_num workers,training_num epochs,training_learning rate,training_positive weight,...,training_recall,training_f1,training_iou,training_specificity,evaluation_accuracy,evaluation_precision,evaluation_recall,evaluation_f1,evaluation_iou,evaluation_specificity
3,norm_shadow_resnet101_256(64)_BCE+DICE0.5_redn...,DeepLabV3,resnet101,256,64,16,16,50,1e-05,10.62,...,0.4630,0.4880,0.3298,0.9973,0.996,0.5097,0.6227,0.5606,0.389,0.9975
4,norm_shadow_resnet101_256(64)_BCE+DICE0.5_redn...,DeepLabV3,resnet101,256,64,16,16,50,1e-05,10.62,...,0.4992,0.4994,0.3409,0.9968,0.996,0.5262,0.5951,0.5585,0.387,0.9978
0,norm_shadow_resnet101_128(32)_BCE+DICE0.5_redn...,DeepLabV3,resnet101,128,32,16,16,50,0.0001,10.62,...,0.5141,0.4972,0.3417,0.9928,0.996,0.5121,0.5951,0.5505,0.380,0.9977
2,norm_shadow_resnet101_256(64)_BCE+DICE0.5_redn...,DeepLabV3,resnet101,256,64,16,16,50,0.0001,10.62,...,0.5397,0.4992,0.3403,0.9960,0.996,0.5428,0.5560,0.5493,0.379,0.9981
1,norm_shadow_resnet101_64(16)_BCE+DICE0.5_redno...,DeepLabV3,resnet101,64,16,8,8,50,0.0001,10.62,...,0.6277,0.3782,0.2531,0.9478,0.996,0.4660,0.5120,0.4879,0.323,0.9976


norm_shadow_resnet101_256(64)_BCE+DICE0.5_rednone_b16w16e50_l1e-5
